# FLUX Model → Google Drive (Colab)

**Download Flux-Apex-V1.flx directly to Google Drive without using Colab's working directory.**

This notebook:
1. Mounts your Google Drive
2. Downloads the model (~17.4GB) directly to Drive
3. Verifies the download
4. Optionally compresses to save Drive space

**Requirements:**
- Google account with ~18GB free Drive space
- HuggingFace account (optional, for faster downloads)

**Why use this?**
- Colab working directory is temporary (deleted after session)
- Google Drive is persistent (model saved across sessions)
- No disk space issues on Colab

In [ ]:
"""Cell 1: Mount Google Drive"""

from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set up paths
DRIVE_ROOT = Path('/content/drive/MyDrive')
FLUX_DIR = DRIVE_ROOT / 'FLUX'
CHECKPOINT_DIR = FLUX_DIR / 'checkpoints'

# Create directories
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Google Drive mounted")
print(f"✓ FLUX directory: {FLUX_DIR}")
print(f"✓ Checkpoints will be saved to: {CHECKPOINT_DIR}")

# Check available space
import shutil
total, used, free = shutil.disk_usage(str(DRIVE_ROOT))
print(f"\n📊 Google Drive Space:")
print(f"   Total: {total / 1e9:.1f} GB")
print(f"   Used:  {used / 1e9:.1f} GB")
print(f"   Free:  {free / 1e9:.1f} GB")

if free / 1e9 < 18:
    print(f"\n⚠️  Warning: Need ~18GB free space. You have {free/1e9:.1f}GB")
else:
    print(f"\n✓ Sufficient space available")

In [ ]:
"""Cell 2: Setup HuggingFace Token (Optional but Recommended)"""

# Option 1: Use Colab secrets (recommended)
# Go to: Runtime → Manage secrets → Add HF_TOKEN

# Option 2: Enter token directly
# Get your token from: https://huggingface.co/settings/tokens

hf_token = None

# Try to get token from Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✓ HuggingFace token loaded from Colab secrets")
except:
    pass

# If no secret, prompt for input
if not hf_token:
    print("No HF_TOKEN found in Colab secrets.")
    print("You can:")
    print("  1. Add HF_TOKEN to Colab secrets (Runtime → Manage secrets)")
    print("  2. Enter token below (optional - downloads still work without it)")
    print("")
    user_input = input("Enter HuggingFace token (or press Enter to skip): ").strip()
    if user_input:
        hf_token = user_input
        print("✓ Token set")
    else:
        print("⚠️  Proceeding without token (may have slower download speeds)")

In [ ]:
"""Cell 3: Download Model Directly to Google Drive"""

from huggingface_hub import hf_hub_download
import time

CHECKPOINT_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'

# Check if already downloaded
if CHECKPOINT_PATH.exists():
    size_gb = CHECKPOINT_PATH.stat().st_size / 1e9
    print(f"✓ Model already exists in Google Drive!")
    print(f"  Path: {CHECKPOINT_PATH}")
    print(f"  Size: {size_gb:.2f} GB")
    print(f"\nTo re-download, delete the file first:")
    print(f"  !rm '{CHECKPOINT_PATH}'")
else:
    print("📥 Downloading Flux-Apex-V1.flx to Google Drive...")
    print("   This may take 10-30 minutes depending on connection speed.")
    print(f"   Destination: {CHECKPOINT_PATH}")
    print("")
    
    start_time = time.time()
    
    # Download directly to Google Drive
    # The key is setting local_dir to the Drive path
    downloaded = hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(FLUX_DIR),
        token=hf_token,
        resume_download=True,  # Resume if interrupted
    )
    
    elapsed = time.time() - start_time
    size_gb = Path(downloaded).stat().st_size / 1e9
    
    print(f"\n✓ Download complete!")
    print(f"  Path: {downloaded}")
    print(f"  Size: {size_gb:.2f} GB")
    print(f"  Time: {elapsed/60:.1f} minutes")
    print(f"  Speed: {size_gb*1024/(elapsed):.1f} MB/s")

In [ ]:
"""Cell 4: Verify Download"""

import hashlib

print("🔍 Verifying download...")

if not CHECKPOINT_PATH.exists():
    print("❌ Error: Model file not found!")
    print(f"   Expected: {CHECKPOINT_PATH}")
else:
    size = CHECKPOINT_PATH.stat().st_size
    size_gb = size / 1e9
    
    # Quick validation: check file size is reasonable
    expected_min = 17.0  # GB
    expected_max = 18.0  # GB
    
    if expected_min <= size_gb <= expected_max:
        print(f"✓ File size OK: {size_gb:.2f} GB")
    else:
        print(f"⚠️  Unexpected file size: {size_gb:.2f} GB")
        print(f"   Expected: {expected_min}-{expected_max} GB")
    
    # Optional: compute partial checksum (first 100MB)
    print("  Computing partial checksum...")
    hasher = hashlib.md5()
    with open(CHECKPOINT_PATH, 'rb') as f:
        hasher.update(f.read(100 * 1024 * 1024))  # First 100MB
    partial_hash = hasher.hexdigest()[:16]
    print(f"✓ Partial MD5: {partial_hash}")
    
    print(f"\n📦 Model ready at: {CHECKPOINT_PATH}")

In [ ]:
"""Cell 5: Optional - Compress to Save Drive Space"""

# Only run this if you want to compress the model to save ~7-8GB Drive space
# After compression, you'll have a .gz file that needs to be decompressed before use

COMPRESS_MODEL = False  # Set to True to compress

if COMPRESS_MODEL:
    import gzip
    import shutil
    
    compressed_path = CHECKPOINT_PATH.with_suffix('.flx.gz')
    
    if compressed_path.exists():
        print(f"✓ Compressed file already exists: {compressed_path}")
        print(f"  Size: {compressed_path.stat().st_size / 1e9:.2f} GB")
    else:
        print("🗜️ Compressing model...")
        print("   This will take 10-20 minutes...")
        
        original_size = CHECKPOINT_PATH.stat().st_size
        
        with open(CHECKPOINT_PATH, 'rb') as f_in:
            with gzip.open(compressed_path, 'wb', compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out, length=64*1024*1024)
        
        compressed_size = compressed_path.stat().st_size
        ratio = (1 - compressed_size / original_size) * 100
        
        print(f"\n✓ Compression complete!")
        print(f"  Original: {original_size / 1e9:.2f} GB")
        print(f"  Compressed: {compressed_size / 1e9:.2f} GB")
        print(f"  Saved: {ratio:.1f}%")
        
        # Ask about deleting original
        delete_original = input("\nDelete original to save space? (y/n): ").strip().lower()
        if delete_original == 'y':
            os.remove(CHECKPOINT_PATH)
            print(f"✓ Deleted original. Compressed file: {compressed_path}")
        else:
            print(f"✓ Kept both files")
else:
    print("ℹ️ Compression skipped (set COMPRESS_MODEL = True to compress)")
    print(f"  Current model size: {CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")

In [ ]:
"""Cell 6: Test Loading the Model"""

# Test that the model can be loaded
# This clones the FLUX repo to get the loader utilities

import subprocess

# Clone FLUX repo for utilities (small, just code)
FLUX_CODE = Path('/content/FLUX')
if not FLUX_CODE.exists():
    print("📦 Cloning FLUX repository for utilities...")
    subprocess.run(['git', 'clone', 'https://github.com/Unseengap/FLUX.git', str(FLUX_CODE)])
    print("✓ Repository cloned")

import sys
sys.path.insert(0, str(FLUX_CODE))

# Test load
print("\n🔍 Testing model load...")
try:
    from flux_model import FLUXModel
    
    # Just load metadata without full weights
    import torch
    raw = torch.load(str(CHECKPOINT_PATH), map_location='cpu', weights_only=False)
    
    print(f"✓ Model loads successfully!")
    print(f"  Format: {raw.get('format', 'unknown')}")
    print(f"  Version: {raw.get('version', 'unknown')}")
    
    if 'metadata' in raw:
        meta = raw['metadata']
        print(f"  Created: {meta.get('created', 'unknown')}")
        
    # Count components
    components = raw.get('components', {})
    enabled = sum(1 for v in components.values() if v)
    print(f"  Components: {enabled} enabled")
    
    # Clean up
    del raw
    import gc
    gc.collect()
    
except Exception as e:
    print(f"❌ Load test failed: {e}")

In [ ]:
"""Cell 7: Usage Instructions"""

print("="*60)
print("  FLUX MODEL SAVED TO GOOGLE DRIVE")
print("="*60)

print(f"""
📁 Location: {CHECKPOINT_PATH}

🔄 To use in future Colab sessions:

1. Mount Google Drive:
   from google.colab import drive
   drive.mount('/content/drive')

2. Set path to model:
   MODEL_PATH = '/content/drive/MyDrive/FLUX/checkpoints/Flux-Apex-V1.flx'

3. Load the model:
   from flux_model import FLUXModel
   model = FLUXModel.load(MODEL_PATH)

💡 Tips:
- The model persists in your Drive across sessions
- No need to re-download each time
- To free space, compress the model (Cell 5)
- To update, delete and re-run Cell 3
""")

# Final space check
total, used, free = shutil.disk_usage(str(DRIVE_ROOT))
print(f"📊 Google Drive Space After Download:")
print(f"   Used:  {used / 1e9:.1f} GB")
print(f"   Free:  {free / 1e9:.1f} GB")

---

## Quick Reference

### Load Model in Future Sessions

```python
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone FLUX for utilities
!git clone https://github.com/Unseengap/FLUX.git /content/FLUX

# Load model
import sys
sys.path.insert(0, '/content/FLUX')
from flux_model import FLUXModel

model = FLUXModel.load('/content/drive/MyDrive/FLUX/checkpoints/Flux-Apex-V1.flx')
```

### If You Compressed the Model

```python
import gzip
import shutil

# Decompress first
with gzip.open('/content/drive/MyDrive/FLUX/checkpoints/Flux-Apex-V1.flx.gz', 'rb') as f_in:
    with open('/content/Flux-Apex-V1.flx', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

# Then load from local
model = FLUXModel.load('/content/Flux-Apex-V1.flx')
```

### Model Info
- **Size**: ~17.4 GB (uncompressed) / ~10 GB (gzip)
- **Parameters**: ~8.34B
- **Version**: 8.1-complete